# Camada Silver - Global Superstore + World Data 2017 (Gabriel)

Limpa as tabelas Bronze e grava as tabelas Silver em Delta, na base `silver` do grupo.

| Tabela Silver | 1 linha = | Origem (Bronze) | Usada no modelo em |
|---|---|---|---|
| `silver.superstore_vendas` | 1 linha de encomenda (`row_id`) com encomenda entre junho e dezembro de 2017 | `superstore_orders` | `fact_vendas_superstore`, `dim_perfil_encomenda`, `dim_localizacao`, `dim_categoria_produto` |
| `silver.world_data_2017` | 1 país | `world_data_2017` | `dim_localizacao` (indicadores de 2017) |

Regras do grupo aplicadas:
- **Intervalo comum:** só entram encomendas de 2017-06-01 a 2017-12-31 (antes a Silver tinha 2014–2017).
- Ficam as colunas de que o modelo precisa: `row_id` e `order_id` (dimensões degeneradas), `quantity`, `ship_mode`,
  `market`, `state` e `region`.
- `discount` é uma taxa entre 0 e 1: chama-se `desconto_pct` (diferente do `desconto_valor` do DataCo).
- Os indicadores booleanos usam o prefixo `is_`, como no modelo.
- Os valores extremos são **marcados** (regra IQR com k = 3, calculada só sobre o intervalo comum), não eliminados.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta import *

# Stack comum do grupo: HDFS (hdfs-nn), Hive Metastore e Delta Lake 2.4
warehouse_location = "hdfs://hdfs-nn:9000/warehouse"

builder = SparkSession \
    .builder \
    .appName("Silver - Global Superstore + World Data (Gabriel)") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.session.timeZone", "UTC") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [2]:
# Intervalo de tempo comum do grupo (README): só entram registos de 1 de junho a 31 de dezembro de 2017
DATA_INICIO = "2017-06-01"
DATA_FIM = "2017-12-31"

In [3]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver LOCATION 'hdfs://hdfs-nn:9000/warehouse/silver.db/'")

def gravar_silver(df, tabela):
    df.write \
        .mode("overwrite") \
        .format("delta") \
        .option("overwriteSchema", "true") \
        .option("path", f"hdfs://hdfs-nn:9000/warehouse/silver.db/{tabela}/") \
        .saveAsTable(f"silver.{tabela}")
    print(f"silver.{tabela}: {spark.table(f'silver.{tabela}').count()} linhas")

def nulos(df):
    """Número de nulos por coluna."""
    df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show(vertical=True)

def texto(c):
    """Remove espaços nas pontas e passa texto vazio a NULL."""
    t = F.trim(F.col(c))
    return F.when(t == "", None).otherwise(t)

## Global Superstore

In [4]:
import re
def to_snake(nome):
    return re.sub(r"[^a-z0-9]+", "_", nome.strip().lower()).strip("_")

bronze = spark.table("bronze.superstore_orders").drop("_data_ingestao", "_ficheiro_origem")
df = bronze.toDF(*[to_snake(c) for c in bronze.columns])
print("Linhas na Bronze:", df.count(), "| colunas:", df.columns)
print("row_id duplicados:", df.count() - df.select("row_id").distinct().count())

Linhas na Bronze: 51290 | colunas: ['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'postal_code', 'city', 'state', 'country', 'region', 'market', 'product_id', 'product_name', 'sub_category', 'category', 'sales', 'quantity', 'discount', 'profit', 'shipping_cost', 'order_priority']
row_id duplicados: 0


In [5]:
def numero(c, tipo="double"):
    t = F.trim(F.col(c))
    return F.when((t == "") | t.isNull(), None).otherwise(t.cast(tipo))

def data(c):
    # O Excel lido com pandas dá "AAAA-MM-DD 00:00:00"
    return F.to_date(F.substring(F.trim(F.col(c)), 1, 10), "yyyy-MM-dd")

vendas = df.dropDuplicates(["row_id"]).select(
    numero("row_id", "int").alias("row_id"),
    texto("order_id").alias("order_id"),
    data("order_date").alias("data_encomenda"),
    data("ship_date").alias("data_envio"),
    texto("ship_mode").alias("modo_envio"),
    texto("market").alias("mercado"),
    texto("region").alias("regiao"),
    texto("country").alias("pais"),
    texto("state").alias("estado"),
    texto("city").alias("cidade"),
    texto("category").alias("categoria"),
    texto("sub_category").alias("subcategoria"),
    texto("product_id").alias("product_id"),
    numero("quantity", "int").alias("quantidade"),
    numero("sales").alias("vendas"),
    numero("discount").alias("desconto_pct"),
    numero("profit").alias("lucro"),
    numero("shipping_cost").alias("custo_envio"),
    texto("order_priority").alias("prioridade_encomenda"))

print("Datas inválidas:", vendas.filter(F.col("data_encomenda").isNull()).count())
vendas.select(F.min("data_encomenda"), F.max("data_encomenda")).show()

antes = vendas.count()
vendas = vendas.filter(F.col("data_encomenda").between(DATA_INICIO, DATA_FIM))
print(f"Removidas {antes - vendas.count()} linhas fora de {DATA_INICIO} a {DATA_FIM}. Ficam {vendas.count()}.")

Datas inválidas: 0
+-------------------+-------------------+
|min(data_encomenda)|max(data_encomenda)|
+-------------------+-------------------+
|         2014-01-01|         2017-12-31|
+-------------------+-------------------+

Removidas 38836 linhas fora de 2017-06-01 a 2017-12-31. Ficam 12454.


In [6]:
# Linhas incoerentes nos campos essenciais (vendas negativas, desconto fora de [0,1], campos em falta)
invalidas = vendas.filter(
    F.col("pais").isNull() | F.col("categoria").isNull() | F.col("subcategoria").isNull() |
    F.col("vendas").isNull() | (F.col("vendas") < 0) | F.col("lucro").isNull() |
    F.col("desconto_pct").isNull() | ~F.col("desconto_pct").between(0, 1) | F.col("quantidade").isNull())
print("Linhas inválidas:", invalidas.count())
vendas = vendas.subtract(invalidas)

# Valores extremos (IQR, k = 3) calculados no intervalo comum: marcados, não eliminados
def limites_iqr(coluna, k=3.0):
    q1, q3 = vendas.approxQuantile(coluna, [0.25, 0.75], 0.001)
    return q1 - k * (q3 - q1), q3 + k * (q3 - q1)

v_lo, v_hi = limites_iqr("vendas")
l_lo, l_hi = limites_iqr("lucro")
print(f"Limites IQR: vendas [{v_lo:.2f}, {v_hi:.2f}] | lucro [{l_lo:.2f}, {l_hi:.2f}]")

vendas = (vendas
    .withColumn("is_vendas_extremas", (F.col("vendas") < v_lo) | (F.col("vendas") > v_hi))
    .withColumn("is_lucro_extremo", (F.col("lucro") < l_lo) | (F.col("lucro") > l_hi))
    .withColumn("is_desconto_alto", F.col("desconto_pct") >= 0.70)
    .withColumn("is_prejuizo", F.col("lucro") < 0))

Linhas inválidas: 0
Limites IQR: vendas [-634.66, 917.27] | lucro [-112.32, 149.76]


In [7]:
# Validações antes de gravar
assert vendas.count() == vendas.select("row_id").distinct().count(), "row_id repetido"
assert vendas.filter(~F.col("data_encomenda").between(DATA_INICIO, DATA_FIM)).count() == 0
nulos(vendas)
vendas.select([F.sum(F.col(c).cast("int")).alias(c) for c in ["is_vendas_extremas", "is_lucro_extremo", "is_desconto_alto", "is_prejuizo"]]).show()
gravar_silver(vendas, "superstore_vendas")

-RECORD 0-------------------
 row_id               | 0   
 order_id             | 0   
 data_encomenda       | 0   
 data_envio           | 0   
 modo_envio           | 0   
 mercado              | 0   
 regiao               | 0   
 pais                 | 0   
 estado               | 0   
 cidade               | 0   
 categoria            | 0   
 subcategoria         | 0   
 product_id           | 0   
 quantidade           | 0   
 vendas               | 0   
 desconto_pct         | 0   
 lucro                | 0   
 custo_envio          | 0   
 prioridade_encomenda | 0   
 is_vendas_extremas   | 0   
 is_lucro_extremo     | 0   
 is_desconto_alto     | 0   
 is_prejuizo          | 0   

+------------------+----------------+----------------+-----------+
|is_vendas_extremas|is_lucro_extremo|is_desconto_alto|is_prejuizo|
+------------------+----------------+----------------+-----------+
|               725|            1417|             472|       3045|
+------------------+---------------

## World Data 2017

In [8]:
wd = spark.table("bronze.world_data_2017")

def num_limpo(c, tipo):
    """Números com separadores de milhares e % ("37,200,000", "11.2%") passam a número; vazio passa a NULL."""
    t = F.trim(F.col(c))
    limpo = F.regexp_replace(t, r"[^0-9.\-]", "")
    return F.when(t.isNull() | (t == "") | (limpo == "") | (limpo == "-"), None).otherwise(limpo.cast(tipo))

world = wd.select(
    texto("Country").alias("pais"),
    texto("Abbreviation").alias("codigo_iso"),
    num_limpo("Year", "int").alias("ano"),
    num_limpo("GDP", "decimal(20,0)").alias("pib_usd"),
    num_limpo("Population", "bigint").alias("populacao"),
    num_limpo("Unemployment rate", "decimal(5,1)").alias("taxa_desemprego_pct"),
    num_limpo("Co2-Emissions", "bigint").alias("emissoes_co2_kt"))

assert world.count() == 195, "Esperavam-se 195 países"
assert world.count() == world.select("pais").distinct().count(), "país repetido"
assert world.filter(F.col("ano") != 2017).count() == 0
assert world.filter(~F.col("taxa_desemprego_pct").between(0, 100)).count() == 0
nulos(world)
gravar_silver(world, "world_data_2017")

-RECORD 0------------------
 pais                | 0   
 codigo_iso          | 7   
 ano                 | 0   
 pib_usd             | 4   
 populacao           | 4   
 taxa_desemprego_pct | 16  
 emissoes_co2_kt     | 3   

silver.world_data_2017: 195 linhas


## Ligação entre as duas tabelas (país)

Países do Superstore que não aparecem com o mesmo nome no World Data. Estes nomes entram na tabela de países do
grupo (normalização para a `dim_localizacao`).

In [9]:
sem_par = (spark.table("silver.superstore_vendas").select("pais").distinct()
    .join(spark.table("silver.world_data_2017").select("pais"), "pais", "left_anti"))
print("Países do Superstore sem correspondência no World Data:", sem_par.count())
sem_par.orderBy("pais").show(50, truncate=False)

Países do Superstore sem correspondência no World Data: 7
+---------------+
|pais           |
+---------------+
|Côte-d'Ivoire  |
|French Guiana  |
|Hong Kong      |
|Ireland        |
|Macedonia      |
|Martinique     |
|Myanmar (Burma)|
+---------------+



## Resultado

In [10]:
spark.sql("""
    SELECT date_format(data_encomenda, 'yyyy-MM') AS mes, COUNT(*) AS linhas, COUNT(DISTINCT order_id) AS encomendas,
           ROUND(SUM(vendas), 0) AS vendas, ROUND(SUM(lucro) / SUM(vendas) * 100, 1) AS margem_pct
    FROM silver.superstore_vendas GROUP BY 1 ORDER BY 1
""").show()
spark.sql("""
    SELECT mercado, COUNT(*) AS linhas, ROUND(SUM(vendas), 0) AS vendas, ROUND(SUM(lucro) / SUM(vendas) * 100, 1) AS margem_pct
    FROM silver.superstore_vendas GROUP BY mercado ORDER BY vendas DESC
""").show()

+-------+------+----------+--------+----------+
|    mes|linhas|encomendas|  vendas|margem_pct|
+-------+------+----------+--------+----------+
|2017-06|  1748|       902|401814.0|      10.9|
|2017-07|  1087|       540|258706.0|      10.8|
|2017-08|  1675|       847|456620.0|      11.7|
|2017-09|  2018|      1019|481157.0|      14.1|
|2017-10|  1626|       810|422767.0|      13.8|
|2017-11|  2147|      1080|555279.0|      11.3|
|2017-12|  2153|      1100|503144.0|       9.3|
+-------+------+----------+--------+----------+

+------------+------+--------+----------+
|     mercado|linhas|  vendas|margem_pct|
+------------+------+--------+----------+
|Asia Pacific|  3408|945499.0|      10.8|
|      Europe|  2961|842587.0|      13.1|
|       LATAM|  2526|547831.0|      10.1|
|        USCA|  2455|546385.0|      12.2|
|      Africa|  1104|197185.0|      13.9|
+------------+------+--------+----------+



In [11]:
spark.stop()